# Lab 1：手写 GPT-2

上节课的结尾是一个单头 attention。这节把它补成完整的 GPT-2，分两步：

1. **形状填空**：每一步的张量和每个权重矩阵是什么形状
2. **代码填空**：把推出来的形状写成代码

我们后面要在 TinyStories 上训练的模型用下面这组配置：12 层、宽度 256、8 个头、上下文 256、词表 4096。它和 GPT-2 small（12 层、768、12 头、1024、50257）的结构**完全一样**，只是数字小。第 3 节会把同一份代码换成 GPT-2 small 的配置，装进 OpenAI 发布的权重，验证你写出了 GPT-2。

填空的地方写的是 `____`。每个代码块下面都有检查，没填完或者填错了会直接告诉你。

In [ ]:
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(torch.__version__, device)

class _Blank:                                   # 填空占位符：被用到时直接报错
    def _fail(self, *a, **k): raise NotImplementedError("这里还有空没填：____")
    __call__ = __getattr__ = __getitem__ = __iter__ = _fail
    __add__ = __radd__ = __sub__ = __rsub__ = __mul__ = __rmul__ = _fail
    __matmul__ = __rmatmul__ = __truediv__ = __rtruediv__ = _fail
____ = _Blank()

---
## 1. 形状填空

记号：$B$ = batch，$T$ = 序列长度（这里取满上下文 256），$C$ = 宽度 `n_embd`，$H$ = 头数，$L$ = 层数，$V$ = 词表大小。

GPT-2 一次前向的全过程：

```
idx (B,T) 整数 token id
 → tok_emb = wte(idx)               查表：每个 id 换成一个 C 维向量
 → pos_emb = wpe(0..T-1)            每个位置也换成一个 C 维向量
 → x = tok_emb + pos_emb
 → 12 个 block，每个：
      x = x + Attention(LN(x))
      x = x + MLP(LN(x))
 → LN → lm_head → logits            每个位置对词表里每个词输出一个概率
```

Attention 里多头的做法：`c_attn` 一次算出 q、k、v矩阵；再把 C 维切成 H 份，每个头只用 C/H 维，各算各的 attention，最后拼回 C 维。

用上面的变量写出形状（元组）。

In [ ]:
B, T, C, H, L, V = 32, 256, 256, 8, 12, 4096

act = {
    "idx":         (B, T),          # 示例
    "tok_emb":     ____,            # wte(idx)
    "pos_emb":     ____,            # wpe(torch.arange(T))（广播前）
    "x":           ____,            # tok_emb + pos_emb（广播后）
    "qkv":         ____,            # c_attn(ln_1(x))，q k v 拼在一起
    "q（k,v）":           ____,            # 把 q 切成多头之后，头放在第 2 维
    "att":         ____,            # q @ k.transpose(-2, -1)：每个头分别算每个位置对每个位置的注意力分数
    "y_heads":     ____,            # att @ v
    "y":           ____,            # 多头拼回去
    "mlp_hidden":  ____,            # MLP 中间层（GPT-2 是 4 倍宽）
    "logits":      ____,            # lm_head 对每个位置的输出，词表logits形式
    "logits_flat": ____,            # 送进 F.cross_entropy 前要将B,T合并拉平成二维
}

权重矩阵的形状。PyTorch 的约定：`nn.Linear(in, out).weight` 的形状是 `(out, in)`，`nn.Embedding(num, dim).weight` 是 `(num, dim)`。

In [ ]:
W = {
    "wte":          ____,           # token embedding
    "wpe":          ____,           # 位置 embedding（最多 T 个位置）
    "attn.c_attn":  ____,           # 每个位置上的C维向量要做矩阵乘法变为q,k,v的纵向拼合，也就是每个位置共输出3C个量
    "attn.c_proj":  ____,           # 多头拼回后的输出投影
    "mlp.c_fc":     ____,           # Feed forward层中间层数是C的4倍
    "mlp.c_proj":   ____,           # 由这个4倍的层再投影回原维度
    "ln_1":         ____,           # LayerNorm 的 逐元素缩放比例（和它形状相同的还有一个偏置）
    "lm_head":      ____,           # 和 wte 共享同一块内存，形状一致（下面 2.3 节解释）
}

# 整个模型一共多少参数？每一层的Attention投影，所有 Linear 和 LayerNorm 都带 bias，lm_head 不带 bias 且和 wte 共享。别忘了最后一个block之后还有一个layer norm才会投影回词表！
# 用上面的变量写一个算式。提示：先算一个 block 有多少，答案形如 a·C² + b·C
n_params = ____ # wte + wpe + 每个 block + ln_f






下面是答案










In [ ]:
def check_shapes():
    hs = C // H
    want_act = {"idx": (B, T), "tok_emb": (B, T, C), "pos_emb": (T, C), "x": (B, T, C),
                "qkv": (B, T, 3 * C), "q": (B, H, T, hs), "att": (B, H, T, T), "y_heads": (B, H, T, hs),
                "y": (B, T, C), "mlp_hidden": (B, T, 4 * C), "logits": (B, T, V), "logits_flat": (B * T, V)}
    want_W = {"wte": (V, C), "wpe": (T, C), "attn.c_attn": (3 * C, C), "attn.c_proj": (C, C),
              "mlp.c_fc": (4 * C, C), "mlp.c_proj": (C, 4 * C), "ln_1": (C,), "lm_head": (V, C)}
    ok = True
    for name, (got, want) in [*((k, (act[k], v)) for k, v in want_act.items()),
                              *((k, (W[k], v)) for k, v in want_W.items())]:
        good = isinstance(got, tuple) and tuple(got) == want
        ok &= good
        print(f"{'✓' if good else '✗'} {name:12s} 你的 {got if isinstance(got, tuple) else '还没填'}")
    want_n = V * C + T * C + L * (12 * C * C + 13 * C) + 2 * C
    good = isinstance(n_params, int) and n_params == want_n
    ok &= good
    print(f"{'✓' if good else '✗'} n_params     你的 {n_params if isinstance(n_params, int) else '还没填'}")
    print("全部正确" if ok else "还有错的，对照上面的流程图再推一遍")

check_shapes()

**想一想**

- `att` 的形状里有两个 $T$。上下文从 256 翻倍到 512，这一个张量变成多大？GPT-2 small 的上下文是 1024，单层的 `att`（batch 32，fp16）占多少显存？
- 参数里 embedding 占多少？如果词表换回 GPT-2 原版的 50257，又占多少？

---
## 2. 代码填空

### 2.1 多头 causal self-attention

和上节课的 `SelfAttention` 比，多了三件事：

1. **多头**：`(B, T, C)` → `view` 成 `(B, T, H, C/H)` → `transpose` 成 `(B, H, T, C/H)`。之后的矩阵乘法对前两维 $(B, H)$ 自动逐个做，每个头互不干扰。
2. **缩放**：除以 $\sqrt{d}$，这里 $d$ 是**每个头**的维度 C/H。
3. **causal mask**：位置 $t$ 只能看 $0..t$。做法是把 `att` 里"未来"的格子在 softmax 之前填成 $-\infty$，softmax 之后它们的权重就是 0。

`self.mask` 是一个下三角全 1 矩阵，已经准备好了。

In [ ]:
@dataclass
class GPTConfig:
    block_size: int = 256       # 最长上下文
    vocab_size: int = 4096
    n_layer: int = 12
    n_head: int = 8
    n_embd: int = 256
# 到时候套GPT的时候，就改这个。

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg.n_head
        self.c_attn = nn.Linear(cfg.n_embd, 3 * cfg.n_embd)
        self.c_proj = nn.Linear(cfg.n_embd, cfg.n_embd)
        mask = torch.tril(torch.ones(cfg.block_size, cfg.block_size))
        self.register_buffer("mask", mask.view(1, 1, cfg.block_size, cfg.block_size), persistent=False) 

    def forward(self, x):
        B, T, C = x.shape
        hs = C // self.n_head
        q, k, v = self.c_attn(x).split(C, dim=2)                      # 各 (B, T, C)

        # 切成多头：(B, T, C) → (B, H, T, hs)，view和transpose的用法自行查阅
        q = ____
        k = ____
        v = ____

        # 算注意力分数并用根号d缩放：(B, H, T, T)
        att = ____

        # 给未来的位置填 -inf。提示：使用Tensor.masked_fill(条件, 值)，条件和self.mask有关
        att = ____

        att = att.softmax(dim=-1)
        self.att = att                                                # 存下来画图用
        y = att @ v                                                   # (B, H, T, hs)

        # 多头拼回去：(B, H, T, hs) → (B, T, C)。提示：transpose 之后要 .contiguous() 才能 view
        y = ____
        return self.c_proj(y)

下面检查你上面的 attention 是否写得正确

In [ ]:
# 检查 1：和 PyTorch 官方的 scaled_dot_product_attention 结果一致
cfg = GPTConfig()
attn = CausalSelfAttention(cfg)
x = torch.randn(2, 10, cfg.n_embd)
with torch.no_grad():
    out = attn(x)
    q, k, v = attn.c_attn(x).split(cfg.n_embd, dim=2) 
    split = lambda t: t.view(2, 10, cfg.n_head, -1).transpose(1, 2)
    ref = F.scaled_dot_product_attention(split(q), split(k), split(v), is_causal=True)
    ref = attn.c_proj(ref.transpose(1, 2).reshape(2, 10, cfg.n_embd))
print("形状", tuple(out.shape), "与官方实现的最大误差", (out - ref).abs().max().item())
assert torch.allclose(out, ref, atol=1e-5), "和官方实现对不上"

# 检查 2：改动第 6 个位置之后的输入，前 6 个位置的输出不能变
x2 = x.clone(); x2[:, 6:] = torch.randn(2, 4, cfg.n_embd)
with torch.no_grad():
    out2 = attn(x2)
assert torch.allclose(out[:, :6], out2[:, :6], atol=1e-6), "前面的位置看到了未来"
print("✓ attention 正确，且不会偷看未来")

### 2.2 MLP 和 Block

MLP 是两层全连接，中间 4 倍宽，激活函数 GELU（ReLU 的平滑版，GPT-2 用的是 tanh 近似）。已经写好。

接下来需要你组装整个block。大体的流程是：输入在进入每个 attention 和 MLP 之前，先进行一次 LayerNorm，残差连接直接把输入加到输出上：

$$x \leftarrow x + \text{Attn}(\text{LN}_1(x)), \qquad x \leftarrow x + \text{MLP}(\text{LN}_2(x))$$


In [ ]:
class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.c_fc = nn.Linear(cfg.n_embd, 4 * cfg.n_embd)
        self.gelu = nn.GELU(approximate="tanh")
        self.c_proj = nn.Linear(4 * cfg.n_embd, cfg.n_embd)
    def forward(self, x):
        return self.c_proj(self.gelu(self.c_fc(x)))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln_1 = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln_2 = nn.LayerNorm(cfg.n_embd)
        self.mlp = MLP(cfg)
    def forward(self, x):
        x = ____
        x = ____
        return x

下面用代码检查一下你写的block对不对

In [ ]:
blk = Block(cfg)
x = torch.randn(2, 10, cfg.n_embd)
with torch.no_grad():
    out = blk(x)
    a = x + blk.attn(blk.ln_1(x))
    ref = a + blk.mlp(blk.ln_2(a))
assert out.shape == x.shape and torch.allclose(out, ref, atol=1e-6), "Block 的结构不对，检查 LN 的位置和残差"
print("✓ Block 正确")

### 2.3 整个 GPT

要填的三处：

1. **embedding**：token embedding 加位置 embedding。位置就是 `0, 1, ..., T-1`
2. **权重共享**：`lm_head` 把 C 维向量变回词表上的分数，形状 `(V, C)`，和 `wte` 一样。GPT-2 让两者直接用**同一个**张量：输入端"这个词对应什么向量"和输出端"这个向量像哪个词"本来就是一回事，还省下 $V \times C$ 个参数
3. **loss**：每个位置预测下一个 token，交叉熵。`F.cross_entropy` 要求输入 `(N, V)` 和目标 `(N,)`，所以先用 `reshape(-1, V)` 和 `reshape(-1)` 拉平（targets 可能是切片出来的、内存不连续，`view` 会报错）

注意，过一个 block 之后：
$$\sigma^2_{x'} = \sigma^2_x + \sigma^2_{attn} + \sigma^2_{mlp}$$
如果每一层的输出方差都和输入在同一量级（这是常见情况，尤其在接近初始化时），那么：
$$\sigma^2_{x^{(L)}} \approx \sigma^2_x + L \cdot (\sigma^2_{attn} + \sigma^2_{mlp})$$
​因为残差流上一共要加 $2L$ 次，每次加的方差应该缩小 $2L$ 倍，即标准差缩小 $\sqrt 2L$倍。加完之后每一层输出的总方差才不会随层数增长。这是上节课"除以 $\sqrt d$"同一个道理。

In [ ]:
class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(cfg.vocab_size, cfg.n_embd),
            # nn.ModuleList 是一个列表，里面的每个元素都会被当作 nn.Module 来注册，从而可以被 nn.Module 的方法(如 .parameters() )所管理。
            h=nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)]),
            ln_f=nn.LayerNorm(cfg.n_embd),
        ))
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)

        # 权重共享：让 lm_head 的 weight 就是 wte 的 weight（同一个 Parameter 对象）
        self.lm_head.weight = ____

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight"):   #这一条的全名应该叫"transformer.h.层数.c_proj.weight"
                    # 写回残差流的投影
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layer))

    def _init_weights(self, m):
        if isinstance(m, nn.Linear): # 若是线性层(MLP里的)，权重初始化为均值为0，标准差为0.02的正态分布，偏置初始化为0
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding): #若是嵌入层，权重初始化为均值为0，标准差为0.02的正态分布
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            # 有人可能问attention的c_attn和c_proj为什么不在这里初始化？因为它们在Block里已经初始化过了，这里再初始化会覆盖掉。
    
    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.cfg.block_size
        pos = torch.arange(T, device=idx.device)

        # token embedding + 位置 embedding
        x = ____

        for block in self.transformer.h:
            x = block(x)
        logits = self.lm_head(self.transformer.ln_f(x))               # (B, T, V)

        loss = None
        if targets is not None:
            # 这里可以用 F.cross_entropy 直接算
            loss = ____
        return logits, loss

    @torch.no_grad() #装饰器，让调用 generate() 时，即推理时不再计算梯度，节省显存和计算量
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.cfg.block_size:]) #只喂最近 block_size 个 token
            logits = logits[:, -1] / temperature           # 模型给每一个位置都会预测一个next token概率，我们只取最后一个位置的输出
            if top_k is not None:
                kth = torch.topk(logits, top_k).values[:, -1:]
                logits = logits.masked_fill(logits < kth, float("-inf"))
            nxt = torch.multinomial(logits.softmax(-1), 1)
            idx = torch.cat([idx, nxt], dim=1)
        return idx

最后这几个步骤：

torch.topk(logits, top_k)：找出 logits 最大的 top_k 个值。

.values[:, -1:]：取这 top_k 个值里最小的那个，形状是 (B, 1)，方便广播。

logits < kth：所有小于这个阈值的 logits​ 被判为 True。

masked_fill(..., float("-inf"))：把这些位置填成 −∞。softmax 时 e^(−∞) = 0，这些词就永远不会被抽到。

In [ ]:
model = GPT(GPTConfig())

# 检查 1：权重共享
assert model.lm_head.weight is model.transformer.wte.weight, "lm_head 和 wte 没有共享"

# 检查 2：你在第 1 节推的权重形状，和代码里实际的一致
real = {"wte": model.transformer.wte.weight, "wpe": model.transformer.wpe.weight,
        "attn.c_attn": model.transformer.h[0].attn.c_attn.weight, "attn.c_proj": model.transformer.h[0].attn.c_proj.weight,
        "mlp.c_fc": model.transformer.h[0].mlp.c_fc.weight, "mlp.c_proj": model.transformer.h[0].mlp.c_proj.weight,
        "ln_1": model.transformer.h[0].ln_1.weight, "lm_head": model.lm_head.weight}
for k, p in real.items():
    print(f"{'✓' if W[k] == tuple(p.shape) else '✗'} {k:12s} 代码里 {tuple(p.shape)}  你推的 {W[k] if isinstance(W[k], tuple) else '还没填'}")

# 检查 3：参数量。parameters() 会自动去掉共享的重复
n_real = sum(p.numel() for p in model.parameters())
print(f"\n参数量：代码里 {n_real:,}，你算的 {n_params if isinstance(n_params, int) else '还没填'}", "✓" if n_real == n_params else "✗")

# 检查 4：初始 loss
idx, tgt = torch.randint(0, 4096, (4, 64)), torch.randint(0, 4096, (4, 64))
logits, loss = model(idx, targets=tgt)
print(f"\nlogits {tuple(logits.shape)}，初始 loss {loss.item():.3f}，ln(V) = {math.log(4096):.3f}")
assert abs(loss.item() - math.log(4096)) < 0.3

**为什么初始 loss 应该是 $\ln V$？** 刚初始化的模型什么都不知道，最合理的输出是对 $V$ 个词一视同仁，每个概率 $1/V$，交叉熵就是 $-\ln(1/V) = \ln V$。初始化的权重很小（std 0.02），logits 接近 0，softmax 之后正好接近均匀分布。


### 2.4 最后一个检查：能不能背下一个 batch

模型写对的话，反复训同一个小 batch，loss 应该很快降到接近 0。这能排除"梯度没传过去"一类的错误。下一个 token 是随机的，所以它只能靠记，记得住说明整条链路是通的。

In [ ]:
torch.manual_seed(0)
model = GPT(GPTConfig()).to(device)
x = torch.randint(0, 4096, (4, 33), device=device)
inp, tgt = x[:, :-1], x[:, 1:]                       # 输入和目标错开一位：用前面的预测下一个
opt = torch.optim.AdamW(model.parameters(), lr=1e-3) # 这是上节课提到的AdamW优化器,还记得他是怎么工作的吗？
for step in range(201):
    _, loss = model(inp, tgt)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 50 == 0:
        print(f"step {step:3d}  loss {loss.item():.4f}")
assert loss.item() < 0.1, "背不下来，检查前面的代码"
print("✓ 能背下来")

---
## 3. 换成 GPT-2 small，装进 OpenAI 的权重

同一份代码，配置换成 GPT-2 small：

| | 我们的小模型 | GPT-2 small |
|---|---|---|
| 层数 $L$ | 12 | 12 |
| 宽度 $C$ | 256 | 768 |
| 头数 $H$ | 8 | 12 |
| 上下文 | 256 | 1024 |
| 词表 $V$ | 4096 | 50257 |

HuggingFace 上的 GPT-2 权重，参数名和我们的一模一样（这份代码就是照着它起的名）。唯一的差别：OpenAI 当年用的是 `Conv1D` 而不是 `nn.Linear`，权重存的是 `(in, out)`，要转置一下。

下面把权重装进**你写的**模型，和 HuggingFace 的官方实现比 logits，然后让你的模型写一段话。（Colab 自带 `transformers`；第一次运行要下载约 500MB）

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Config, GPT2TokenizerFast

try:
    hf = GPT2LMHeadModel.from_pretrained("gpt2")
    tok = GPT2TokenizerFast.from_pretrained("gpt2")
except Exception as e:                                   # 下载不了时退而求其次：结构一样，权重随机
    print("下载失败，用随机权重的 GPT-2 结构代替：", repr(e)[:100])
    hf, tok = GPT2LMHeadModel(GPT2Config()), None
hf.eval()

def load_from_hf(model, hf):
    sd, hf_sd = model.state_dict(), hf.state_dict()
    keys = [k for k in hf_sd if not k.endswith((".attn.bias", ".attn.masked_bias"))]   # 老版本里的 mask buffer
    assert set(keys) == set(sd), set(keys) ^ set(sd)
    conv1d = ("attn.c_attn.weight", "attn.c_proj.weight", "mlp.c_fc.weight", "mlp.c_proj.weight")
    with torch.no_grad():
        for k in keys:
            sd[k].copy_(hf_sd[k].t() if k.endswith(conv1d) else hf_sd[k])
    return model

gpt2 = load_from_hf(GPT(GPTConfig(block_size=1024, vocab_size=50257, n_layer=12, n_head=12, n_embd=768)), hf).eval()
print(f"参数量 {sum(p.numel() for p in gpt2.parameters()):,}")

idx = torch.randint(0, 50257, (2, 64))
with torch.no_grad():
    diff = (gpt2(idx)[0] - hf(idx).logits).abs().max().item()
print(f"你的模型和 HuggingFace 官方实现的 logits 最大误差：{diff:.2e}")
assert diff < 1e-3
print("✓ 你写的就是 GPT-2")

In [ ]:
if tok is not None:
    torch.manual_seed(0)
    prompt = tok("The meaning of life is", return_tensors="pt").input_ids
    for T_ in [0.7, 1.0]:
        out = gpt2.generate(prompt, max_new_tokens=40, temperature=T_, top_k=50)
        print(f"--- temperature {T_} ---\n{tok.decode(out[0])}\n")

**可以试试**

- 改 `temperature`：0.1 和 2.0 分别会怎样？
- 把 `generate` 改成每步取概率最大的词（greedy），会出现什么问题？
- 把 `load_from_hf` 里的转置去掉，误差变成多少？